In [ ]:
import copy
import os
import random
import time
from contextlib import nullcontext
import numpy as np
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset, WeightedRandomSampler
from torchvision import datasets, transforms
from torchvision.transforms import InterpolationMode
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix,
    roc_curve,
    auc,
    classification_report,
    balanced_accuracy_score,
    precision_recall_fscore_support,
    precision_recall_curve,
    average_precision_score
)
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt

# Fixação de semente para reprodutibilidade
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Treinando em: {device}")

DATA_DIR = "../dataset/preprocess_imgs"
MODELS_DIR = "../models"
RUNS_DIR = "../runs/FusNet_run"
MODEL_SAVE_PATH = os.path.join(MODELS_DIR, "multi_fusnet_best.pth")
TEMP_MODEL_PATH = os.path.join(MODELS_DIR, "temp_model.pth")

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(RUNS_DIR, exist_ok=True)

# Hiperparâmetros de treino
BATCH_SIZE = 32
EPOCHS = 80
LEARNING_RATE = 0.0015     
WARMUP_PCT = 0.10          
WEIGHT_DECAY = 0.01       
GRAD_CLIP = 1.0            
EMA_DECAY = 0.995          
PATIENCE = 10              
BALANCE_POWER = 1.0        
TTA = True                 

NUM_WORKERS = 0 if os.name == "nt" else min(4, os.cpu_count() or 1)

def amp_context():
    return nullcontext()

Treinando em: cuda


### Funções do Modelo

In [8]:
# 1. Improved Huber Loss
class ArticleHuberLoss(nn.Module):
    def __init__(self, delta=1.0):
        super().__init__()
        self.delta = delta

    def forward(self, pred_logits, target):
        pred_probs = torch.softmax(pred_logits, dim=-1)
        target_one_hot = F.one_hot(target, num_classes=pred_logits.size(-1)).float()
        
        error1 = torch.abs(pred_probs - target_one_hot)
        
        quadratic = 0.5 * (error1 ** 2)
        linear = self.delta * error1 - 0.5 * (self.delta ** 2)
        
        loss = torch.where(error1 <= self.delta, quadratic, linear)
        return loss.mean()

# 2. Componentes da Multi-FusNet
class ResidualChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y.expand_as(x)

class ResidualGroup(nn.Module):
    def __init__(self, channels, num_rca=3):
        super().__init__()
        # Pilha de blocos RCA
        self.rca_blocks = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(channels, channels, kernel_size=3, padding=1),
                nn.ReLU(inplace=True),
                nn.Conv2d(channels, channels, kernel_size=3, padding=1),
                ResidualChannelAttention(channels)
            )
            for _ in range(num_rca)
        ])
        self.w_rg = nn.Conv2d(channels, channels, kernel_size=3, padding=1)

    def forward(self, x):
        res = x
        out = x
        for rca_block in self.rca_blocks:
            out = out + rca_block(out)
        return res + self.w_rg(out)

class MultiFusNetCNN(nn.Module):
    def __init__(self, num_classes=3, num_rca=3):
        super().__init__()
        # Extração de características
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        # Multipath Residual
        self.rg_path1_1 = ResidualGroup(64, num_rca=num_rca)
        self.rg_path1_2 = ResidualGroup(64, num_rca=num_rca)
        self.rg_path2 = ResidualGroup(64, num_rca=num_rca)

        # Pixel Shuffle Fusion
        self.ps_high = nn.Sequential(
            nn.Conv2d(64, 64 * 16, kernel_size=3, padding=1),
            nn.PixelShuffle(4)
        )
        self.ps_low = nn.Sequential(
            nn.Conv2d(16, 64 * 4, kernel_size=3, stride=2, padding=1),
            nn.PixelShuffle(2)
        )
        self.fusion_conv = nn.Conv2d(64, 64, kernel_size=3, padding=1)

        # Classificador
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(64, num_classes)

    def forward(self, x):
        low_level = self.conv1(x)
        x_mid = self.conv2(low_level)
        high_level = self.conv3(x_mid)

        path1 = self.rg_path1_2(self.rg_path1_1(high_level))
        path2 = self.rg_path2(high_level)
        multi_out = path1 + path2

        multi_up = self.ps_high(multi_out)
        low_up = self.ps_low(low_level)
        fused = self.fusion_conv(multi_up + low_up)

        out = self.global_pool(fused)
        out = out.view(out.size(0), -1)
        out = self.fc(out)
        
        return out

### Divisão do Dataset (70/10/20)

In [ ]:
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomAffine(
        degrees=20, translate=(0.06, 0.06), scale=(0.9, 1.2),
        interpolation=InterpolationMode.BILINEAR
    ),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.2),
    transforms.ToTensor()
])
eval_transform = transforms.Compose([
    transforms.ToTensor()
])

full_dataset = datasets.ImageFolder(root=DATA_DIR, transform=train_transform)
eval_full_dataset = datasets.ImageFolder(root=DATA_DIR, transform=eval_transform)
num_classes = len(full_dataset.classes)

print(f"Classes encontradas: {full_dataset.classes}")
print(f"Total de imagens no dataset: {len(full_dataset)}")

targets = full_dataset.targets
indices = list(range(len(full_dataset)))

# 1. Separamos 20% para o Teste final e 80% para uma base Temporária (Treino + Validação)
temp_idx, test_idx, temp_targets, test_targets = train_test_split(
    indices, targets, 
    test_size=0.20, 
    stratify=targets, 
    random_state=SEED
)

# 2. Dos 80% restantes, separamos 12.5% (que equivale a exatos 10% do total geral) para Validação
train_idx, val_idx = train_test_split(
    temp_idx, 
    test_size=0.125, 
    stratify=temp_targets, 
    random_state=SEED
)

train_dataset = Subset(full_dataset, train_idx)
val_dataset = Subset(eval_full_dataset, val_idx)
test_dataset = Subset(eval_full_dataset, test_idx)

# Amostragem balanceada por classe, somente no treino (a perda do artigo fica intacta)
train_targets = np.array(targets)[train_idx]
class_counts = np.bincount(train_targets, minlength=num_classes)
sample_weights = (1.0 / class_counts[train_targets]) ** BALANCE_POWER
sampler = WeightedRandomSampler(
    weights=torch.as_tensor(sample_weights, dtype=torch.double),
    num_samples=len(train_dataset),
    replacement=True,
    generator=torch.Generator().manual_seed(SEED)
)

loader_kwargs = dict(
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
    persistent_workers=(NUM_WORKERS > 0)
)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, drop_last=True, **loader_kwargs)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, **loader_kwargs)

def _contagem(idx):
    c = np.bincount(np.array(targets)[idx], minlength=num_classes)
    return {full_dataset.classes[i]: int(c[i]) for i in range(num_classes)}

expected_share = np.bincount(train_targets, weights=sample_weights, minlength=num_classes)
expected_share = expected_share / expected_share.sum()
expected_share_dict = {full_dataset.classes[i]: round(float(expected_share[i]), 3) for i in range(num_classes)}

print(f"\n[DIVISÃO CONCLUÍDA]")
print(f"Treino: {len(train_dataset)} | Validação: {len(val_dataset)} | Teste: {len(test_dataset)}")
print(f"  Treino    por classe: {_contagem(train_idx)}")
print(f"  Validação por classe: {_contagem(val_idx)}")
print(f"  Teste     por classe: {_contagem(test_idx)}")
print(f"  Fração esperada por classe em cada época de treino (amostrador): {expected_share_dict}")

Classes encontradas: ['healthy', 'red_spider_mite', 'rust']
Total de imagens no dataset: 1426

[DIVISÃO CONCLUÍDA]
Treino: 997 | Validação: 143 | Teste: 286
  Treino    por classe: {'healthy': 507, 'red_spider_mite': 109, 'rust': 381}
  Validação por classe: {'healthy': 73, 'red_spider_mite': 15, 'rust': 55}
  Teste     por classe: {'healthy': 146, 'red_spider_mite': 31, 'rust': 109}
  Fração esperada por classe em cada época de treino (amostrador): {'healthy': 0.333, 'red_spider_mite': 0.333, 'rust': 0.333}


### Treinamento

In [ ]:
# Inicialização de pesos Kaiming (convoluções) e BatchNorm
def init_weights(m):
    if isinstance(m, nn.Conv2d):
        nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
        if m.bias is not None:
            nn.init.constant_(m.bias, 0)
    elif isinstance(m, nn.BatchNorm2d):
        nn.init.constant_(m.weight, 1)
        nn.init.constant_(m.bias, 0)

# Instancia o modelo
model = MultiFusNetCNN(num_classes=num_classes).to(device)
model.apply(init_weights)

# Só o classificador final usa N(0, 0.01); os FC do Channel Attention mantêm a inicialização padrão do PyTorch
nn.init.normal_(model.fc.weight, 0, 0.01)
nn.init.constant_(model.fc.bias, 0)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parâmetros treináveis: {n_params / 1e6:.2f} M")

# Improved Huber loss do artigo (Eqs. 16-17, delta = 1.0)
criterion = ArticleHuberLoss(delta=1.0)

# AdamW (sem weight decay em bias/BatchNorm) + OneCycle (warmup + decaimento cosseno, atualizado a cada batch)
decay_params = [p for p in model.parameters() if p.requires_grad and p.ndim > 1]
no_decay_params = [p for p in model.parameters() if p.requires_grad and p.ndim <= 1]
optimizer = torch.optim.AdamW(
    [{'params': decay_params, 'weight_decay': WEIGHT_DECAY},
     {'params': no_decay_params, 'weight_decay': 0.0}],
    lr=LEARNING_RATE
)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=LEARNING_RATE,
    total_steps=EPOCHS * len(train_loader),
    pct_start=WARMUP_PCT,
    anneal_strategy='cos',
    div_factor=25.0,
    final_div_factor=1000.0
)

# Média móvel exponencial (EMA) dos pesos e dos buffers do BatchNorm.
class ModelEMA:
    def __init__(self, model, decay=0.995):
        self.ema = copy.deepcopy(model).eval()
        for p in self.ema.parameters():
            p.requires_grad_(False)
        self.decay = decay
        self.updates = 0

    @torch.no_grad()
    def update(self, model):
        self.updates += 1
        d = min(self.decay, (1 + self.updates) / (10 + self.updates))
        model_state = model.state_dict()
        for k, v in self.ema.state_dict().items():
            if v.dtype.is_floating_point:
                v.mul_(d).add_(model_state[k].detach(), alpha=1.0 - d)
            else:
                v.copy_(model_state[k])

ema = ModelEMA(model, decay=EMA_DECAY)

Parâmetros treináveis: 1.47 M


In [ ]:
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'val_bal_acc': [], 'lr': []}
best_key = (-1.0, -float('inf'))
best_val_loss = float('inf')
best_bal_acc = 0.0
best_epoch = 0
epochs_no_improve = 0

val_size = len(val_loader.dataset)

for epoch in range(EPOCHS):
    t0 = time.time()
    model.train()
    running_loss = 0.0
    train_correct = 0
    seen = 0

    for images, labels in train_loader:
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        scheduler.step()
        ema.update(model)

        bs = images.size(0)
        running_loss += loss.item() * bs
        train_correct += (outputs.argmax(dim=1) == labels).sum().item()
        seen += bs

    epoch_train_loss = running_loss / seen
    epoch_train_acc = train_correct / seen

    # Validação (com os pesos EMA)
    ema.ema.eval()
    val_loss_sum = 0.0
    val_correct = 0
    val_preds, val_labels = [], []

    with torch.no_grad():
        for images, labels in val_loader:
            labels = labels.to(device, non_blocking=True)
            outputs = ema.ema(images)
            loss = criterion(outputs, labels)
            val_loss_sum += loss.item() * images.size(0)

            preds = outputs.argmax(dim=1)
            val_correct += (preds == labels).sum().item()
            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(labels.cpu().numpy())

    epoch_val_loss = val_loss_sum / val_size
    epoch_val_acc = val_correct / val_size
    epoch_val_bal_acc = float(balanced_accuracy_score(val_labels, val_preds))

    current_lr = optimizer.param_groups[0]['lr']

    history['train_loss'].append(epoch_train_loss)
    history['train_acc'].append(epoch_train_acc)
    history['val_loss'].append(epoch_val_loss)
    history['val_acc'].append(epoch_val_acc)
    history['val_bal_acc'].append(epoch_val_bal_acc)
    history['lr'].append(current_lr)

    print(
        f"Epoch [{epoch+1:02d}/{EPOCHS}] "
        f"Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.4f} | "
        f"Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc:.4f} | "
        f"Val BalAcc: {epoch_val_bal_acc:.4f} | LR: {current_lr:.6f} | {time.time() - t0:.0f}s"
    )

    # Seleção do melhor checkpoint e EarlyStop (histórico completo mantido para os gráficos)
    key = (epoch_val_bal_acc, -epoch_val_loss)
    if key > best_key:
        best_key = key
        best_val_loss = epoch_val_loss
        best_bal_acc = epoch_val_bal_acc
        best_epoch = epoch
        epochs_no_improve = 0
        torch.save(ema.ema.state_dict(), TEMP_MODEL_PATH)
        print(f"  -> Melhoria detectada! Val BalAcc: {best_bal_acc:.4f} | Val Loss: {best_val_loss:.4f} (Patience: 0/{PATIENCE})")
    else:
        epochs_no_improve += 1
        print(f"  -> Sem melhoria. (Patience: {epochs_no_improve}/{PATIENCE})")

        if epochs_no_improve >= PATIENCE:
            print(f"\n[EARLY STOPPING ATIVADO] O critério não melhorou nas últimas {PATIENCE} épocas.")
            break

print(f"\nMelhor época: {best_epoch + 1} | Val BalAcc: {best_bal_acc:.4f} | Val Loss: {best_val_loss:.4f}")

if os.path.exists(TEMP_MODEL_PATH):
    os.replace(TEMP_MODEL_PATH, MODEL_SAVE_PATH)
    print(f"\n[SUCESSO] Melhor modelo salvo em: {MODEL_SAVE_PATH}")
    model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=device))

Epoch [01/80] Train Loss: 0.1082 | Train Acc: 0.4183 | Val Loss: 0.0996 | Val Acc: 0.5315 | Val BalAcc: 0.4177 | LR: 0.000115 | 24s
  -> Melhoria detectada! Val BalAcc: 0.4177 | Val Loss: 0.0996 (Patience: 0/25)
Epoch [02/80] Train Loss: 0.1003 | Train Acc: 0.5222 | Val Loss: 0.0959 | Val Acc: 0.5455 | Val BalAcc: 0.4842 | LR: 0.000273 | 23s
  -> Melhoria detectada! Val BalAcc: 0.4842 | Val Loss: 0.0959 (Patience: 0/25)
Epoch [03/80] Train Loss: 0.0948 | Train Acc: 0.5504 | Val Loss: 0.0867 | Val Acc: 0.6084 | Val BalAcc: 0.5463 | LR: 0.000508 | 23s
  -> Melhoria detectada! Val BalAcc: 0.5463 | Val Loss: 0.0867 (Patience: 0/25)
Epoch [04/80] Train Loss: 0.1005 | Train Acc: 0.4960 | Val Loss: 0.0902 | Val Acc: 0.5594 | Val BalAcc: 0.5110 | LR: 0.000785 | 23s
  -> Sem melhoria. (Patience: 1/25)
Epoch [05/80] Train Loss: 0.0985 | Train Acc: 0.5071 | Val Loss: 0.0847 | Val Acc: 0.6084 | Val BalAcc: 0.5873 | LR: 0.001061 | 23s
  -> Melhoria detectada! Val BalAcc: 0.5873 | Val Loss: 0.0847 (

### Gráficos

In [ ]:
# GRÁFICOS DE TREINAMENTO (HISTÓRICO)
epochs = range(1, len(history['train_loss']) + 1)  # [CORRIGIDO] history agora é completo (sem corte)
train_loss = history['train_loss']
val_loss = history['val_loss']
val_acc = history['val_acc']
train_acc = history.get('train_acc', None) 

# 1. Accuracy vs. epoch (Treino vs. Validação)
plt.figure(figsize=(6, 5))
if train_acc is not None and len(train_acc) > 0:
    plt.plot(epochs, train_acc, color='tab:blue', label='Training accuracy', linewidth=1.5)
plt.plot(epochs, val_acc, color='tab:orange', label='Validation accuracy', linewidth=1.5)
if 'val_bal_acc' in history and len(history['val_bal_acc']) > 0:
    plt.plot(epochs, history['val_bal_acc'], color='tab:green', linestyle=':', label='Validation balanced accuracy', linewidth=1.5)
plt.axvline(best_epoch + 1, color='gray', linestyle='--', linewidth=1.0, label=f'Best epoch ({best_epoch + 1})')
plt.xlabel('Epoch', fontweight='bold', fontsize=12)
plt.ylabel('Accuracy', fontweight='bold', fontsize=12)
plt.legend(loc='lower right', fontsize=11)
plt.savefig(os.path.join(RUNS_DIR, "accuracy_vs_epoch.png"), dpi=300, bbox_inches='tight')
plt.close() 

# 2. Loss vs. epoch
plt.figure(figsize=(6, 5))
plt.plot(epochs, train_loss, color='tab:blue', label='Training loss', linewidth=1.5)
plt.plot(epochs, val_loss, color='tab:orange', label='Validation loss', linewidth=1.5)
plt.axvline(best_epoch + 1, color='gray', linestyle='--', linewidth=1.0, label=f'Best epoch ({best_epoch + 1})')
plt.xlabel('Epoch', fontweight='bold', fontsize=12)
plt.ylabel('Loss', fontweight='bold', fontsize=12)
plt.legend(loc='upper right', fontsize=11)
plt.savefig(os.path.join(RUNS_DIR, "loss_vs_epoch.png"), dpi=300, bbox_inches='tight')
plt.close()

# INFERÊNCIA NO CONJUNTO DE TESTE (modelo carregado com os pesos do melhor epoch)
@torch.no_grad()
def predict_probs(net, loader, use_tta=False):
    net.eval()
    probs_all, labels_all = [], []
    for images, labels in loader:
        with amp_context():
            probs = torch.softmax(net(images).float(), dim=1)
            if use_tta:
                # TTA: média das probabilidades com flip horizontal, vertical e ambos
                views = [
                    torch.flip(images, dims=[3]),
                    torch.flip(images, dims=[2]),
                    torch.flip(images, dims=[2, 3]),
                ]
                for v in views:
                    probs = probs + torch.softmax(net(v).float(), dim=1)
                probs = probs / (1 + len(views))
        probs_all.append(probs.cpu().numpy())
        labels_all.append(labels.numpy())
    return np.concatenate(probs_all), np.concatenate(labels_all)

probs_plain, all_labels = predict_probs(model, test_loader, use_tta=False)
probs_tta, _ = predict_probs(model, test_loader, use_tta=True)

def _resumo_rapido(nome, probs):
    preds = probs.argmax(axis=1)
    return f"{nome}: Acc {(preds == all_labels).mean():.4f} | BalAcc {balanced_accuracy_score(all_labels, preds):.4f}"

print(_resumo_rapido("Teste sem TTA", probs_plain))
print(_resumo_rapido("Teste com TTA", probs_tta))
print(f"[O relatório abaixo usa TTA = {TTA}]\n")

all_probs = probs_tta if TTA else probs_plain
all_preds = all_probs.argmax(axis=1)

class_names = eval_full_dataset.classes 
n_classes = len(class_names)

# 3. Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names,
            annot_kws={"size": 16}, cbar_kws={'shrink': 0.8})

plt.xlabel('Predicted Labels', fontweight='bold', fontsize=12)
plt.ylabel('True Labels', fontweight='bold', fontsize=12)
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.savefig(os.path.join(RUNS_DIR, "confusion_matrix.png"), dpi=300, bbox_inches='tight')
plt.close()

# Binarização dos rótulos para curvas multiclasse (ROC e Precision-Recall)
y_test_bin = label_binarize(all_labels, classes=range(n_classes))
colors = sns.color_palette("husl", n_classes)

# 4. ROC Curve
plt.figure(figsize=(6, 5))
roc_auc_dict = {}

if n_classes == 2:
    fpr, tpr, _ = roc_curve(all_labels, all_probs[:, 1])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, color='tab:orange', lw=2, label=f'ROC Curve (AUC = {roc_auc:.4f})')
else:
    for i, color in zip(range(n_classes), colors):
        fpr, tpr, _ = roc_curve(y_test_bin[:, i], all_probs[:, i])
        roc_auc = auc(fpr, tpr)
        roc_auc_dict[class_names[i]] = roc_auc
        plt.plot(fpr, tpr, color=color, lw=1.5, label=f'{class_names[i]} (AUC = {roc_auc:.4f})')

plt.plot([0, 1], [0, 1], linestyle='--', lw=1.5, color='gray')
plt.xlabel('False Positive Rate', fontweight='bold', fontsize=12)
plt.ylabel('True Positive Rate', fontweight='bold', fontsize=12)
plt.legend(loc="lower right", fontsize=10)
plt.savefig(os.path.join(RUNS_DIR, "roc_curve.png"), dpi=300, bbox_inches='tight')
plt.close()

# 5. Precision-Recall Curve
plt.figure(figsize=(6, 5))
pr_auc_dict = {}

if n_classes > 2:
    for i, color in zip(range(n_classes), colors):
        precision, recall, _ = precision_recall_curve(y_test_bin[:, i], all_probs[:, i])
        ap_score = average_precision_score(y_test_bin[:, i], all_probs[:, i])
        pr_auc_dict[class_names[i]] = ap_score
        plt.plot(recall, precision, color=color, lw=1.5, label=f'{class_names[i]} (AP = {ap_score:.4f})')

    plt.xlabel('Recall (Sensibilidade)', fontweight='bold', fontsize=12)
    plt.ylabel('Precision (Precisão)', fontweight='bold', fontsize=12)
    plt.legend(loc="lower left", fontsize=10)
    plt.savefig(os.path.join(RUNS_DIR, "pr_curve.png"), dpi=300, bbox_inches='tight')
    plt.close()

# 6. RELATÓRIO DE MÉTRICAS REAIS NO CONJUNTO DE TESTE
bal_acc = balanced_accuracy_score(all_labels, all_preds)
report_str = classification_report(all_labels, all_preds, target_names=class_names, digits=4)

specificities = []
for i in range(n_classes):
    tp = cm[i, i]
    fp = cm[:, i].sum() - tp
    fn = cm[i, :].sum() - tp
    tn = cm.sum() - (tp + fp + fn)
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    specificities.append(spec)

macro_specificity = np.mean(specificities)
prec_m, rec_m, f1_m, _ = precision_recall_fscore_support(all_labels, all_preds, average='macro', zero_division=0)

resumo_metricas = (
    "============================================================\n"
    "          AVALIAÇÃO REAL NO CONJUNTO DE TESTE (FusNet)      \n"
    "============================================================\n"
    f"Acurácia Global (Padrão):     {(all_preds == all_labels).mean():.4f}\n"
    f"Acurácia Balanceada (Real):   {bal_acc:.4f}\n"
    f"Especificidade Média (Macro): {macro_specificity:.4f}\n"
    f"Precisão / Recall / F1 (Macro): {prec_m:.4f} / {rec_m:.4f} / {f1_m:.4f}\n\n"
    "--- Relatório Detalhado por Classe ---\n"
    f"{report_str}\n"
    "--- Especificidade, ROC-AUC e PR-AUC por Classe ---\n"
)

for idx, cls_name in enumerate(class_names):
    r_auc = roc_auc_dict.get(cls_name, 0.0)
    p_auc = pr_auc_dict.get(cls_name, 0.0)
    spec_cls = specificities[idx]
    resumo_metricas += (
        f" - {cls_name:<15} | Spec: {spec_cls:.4f} | "
        f"ROC-AUC: {r_auc:.4f} | PR-AUC (AP): {p_auc:.4f}\n"
    )

resumo_metricas += "============================================================\n"

print(resumo_metricas)

report_path = os.path.join(RUNS_DIR, "relatorio_metricas.txt")
with open(report_path, "w", encoding="utf-8") as f:
    f.write(resumo_metricas)

print(f"[SUCESSO] Gráficos e relatório de métricas salvos em: {RUNS_DIR}")

Teste sem TTA: Acc 0.8497 | BalAcc 0.7816
Teste com TTA: Acc 0.8566 | BalAcc 0.7931
[O relatório abaixo usa TTA = True]

          AVALIAÇÃO REAL NO CONJUNTO DE TESTE (FusNet)      
Acurácia Global (Padrão):     0.8566
Acurácia Balanceada (Real):   0.7931
Especificidade Média (Macro): 0.9287
Precisão / Recall / F1 (Macro): 0.7741 / 0.7931 / 0.7806

--- Relatório Detalhado por Classe ---
                 precision    recall  f1-score   support

        healthy     0.9404    0.9726    0.9562       146
red_spider_mite     0.5263    0.6452    0.5797        31
           rust     0.8557    0.7615    0.8058       109

       accuracy                         0.8566       286
      macro avg     0.7741    0.7931    0.7806       286
   weighted avg     0.8632    0.8566    0.8581       286

--- Especificidade, ROC-AUC e PR-AUC por Classe ---
 - healthy         | Spec: 0.9357 | ROC-AUC: 0.9846 | PR-AUC (AP): 0.9833
 - red_spider_mite | Spec: 0.9294 | ROC-AUC: 0.9104 | PR-AUC (AP): 0.6169
 - rust 